[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PTB-MR/qMRData/blob/main/quantitative_imaging_molli.ipynb)

In [ ]:
import importlib

if not importlib.util.find_spec("mrpro"):
    %pip install mrpro[notebooks]

## Quanitative Image Reconstruction - T1 Mapping using MOLLI Sequence

In [ ]:
import numpy as np
import torch
from mrpro.algorithms.reconstruction import DirectReconstruction
from mrpro.data import KData, IData, CsmData
from mrpro.data.traj_calculators import KTrajectoryCartesian
from mrpro.operators import DictionaryMatchOp
from mrpro.operators.models import MOLLI
from cmap import Colormap

## Select the raw data files

In [ ]:
raw_file = "/Users/kolbit01/Documents/Data/Claudia_055/Test 03 DB Knee Protocol/Raw/Left Knee/meas_MID00127_FID21425_t1map_MOLLI_Left.mrd"
image_folder = "/Users/kolbit01/Documents/Data/Claudia_055/Test 03 DB Knee Protocol/Dicom/Left Knee/T1Map MOLLI/t1map_MOLLI_Left_T1_map"

vmax = 1.0
ti_times = 0.340 + torch.as_tensor([0, 1, 2, 3, 0, 1, 2, 0, 1, 0, 1])

## Load Dicom

In [ ]:
t1_map_dicom = IData.from_dicom_folder(image_folder)

# Dicom saves T1 values in ms, we use seconds here.
t1_map_dicom.data *= 1e-3

## Reconstruct images

In [ ]:
kdata = KData.from_file(raw_file, trajectory=KTrajectoryCartesian())

# for multi-slice acquisition: resort the data into correct order of slices and contrasts
if (nslices := kdata.header.acq_info.idx.slice.unique().numel()) > 1:
    sort_idx = torch.argsort(kdata.header.acq_info.position.x.squeeze(), stable=True)
    kdata = kdata[sort_idx].rearrange(
        "(slice contrast) ... -> contrast slice ...", slice=nslices
    )

# We have to calculate the coil maps from one of the contrasts which ideally has high signal for all tissue types
csm = CsmData.from_kdata_inati(kdata[0])
recon = DirectReconstruction(kdata, csm=csm)
idata = recon(kdata)

## Create signal model for inversion recovery sequence and calculate quantitative maps

We will select the correct signal model and then create a dictionary mapping operator, similarly what is used in MR Fingerprinting. 
We can then simply apply the dictionary mapping operator to the reconstructed images and obtain the quantitative maps

In [ ]:
dictionary = DictionaryMatchOp(
    MOLLI(ti=torch.as_tensor(ti_times, dtype=torch.float32)),
    index_of_scaling_parameter=0,
)
dictionary.append(
    torch.tensor(1.0),
    torch.linspace(0.1, 3.0, 1000)[None, :],
    torch.linspace(0.1, 5.0, 1000)[None, None, :],
)
m0_match, c_match, t1_match = dictionary(idata.data)
t1_map = IData(t1_match, header=idata.header[0])

## Visualise results

In [ ]:
import matplotlib.pyplot as plt
from einops import rearrange
import nibabel as nib

from nibabel.orientations import (
    io_orientation,
    axcodes2ornt,
    ornt_transform,
    apply_orientation,
)


def show_image(qmap: IData, cmap, vmax: float) -> None:
    """Show the qualitative images."""
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))

    for cax in ax.flatten():
        cax.set_axis_off()

    def orient_images(idata: IData) -> np.array:
        # Ensure the slices are in the correct order
        if idata.shape[0] > 1:
            sort_idx = torch.argsort(idata.header.position.x.squeeze())
            idata = idata[sort_idx]
        orientation = idata.header.orientation.as_matrix().squeeze()
        affine_zyx = torch.cat(
            [
                torch.tensor([[1.0, 0.0, 0.0, 0.0]]),
                torch.cat([torch.zeros((3, 1)), orientation], 1),
            ],
            0,
        )

        data = rearrange(idata.data, "... other z y x-> x y z 1 other (...)")
        img_nii = nib.Nifti1Image(
            data.squeeze().abs().numpy(force=True),
            affine_zyx.flip([0, 1]),
            dtype=np.float32,
        )
        # Target orientation (RAS)
        ras_ornt = axcodes2ornt(("R", "S", "A"))
        transform = ornt_transform(io_orientation(img_nii.affine), ras_ornt)
        ras_data = apply_orientation(img_nii.get_fdata(), transform)
        return ras_data

    def plot_multi_slice_image(ax, img, colorbar_label, cmap, vmax):
        """Plot three slices of M2D/3D image."""
        # Ensure the slices are in the correct order
        idim = img.squeeze().shape[0]
        for cax, slice_idx in zip(
            ax, [int(idim // 2 - idim // 6), idim // 2, int(idim // 2 + idim // 6)]
        ):
            im = cax.imshow(
                np.squeeze(img[slice_idx, :, :]),
                cmap=cmap,
                vmin=0,
                vmax=vmax,
                origin="lower",
            )
            fig.colorbar(im, ax=cax, label=colorbar_label)

    plot_multi_slice_image(ax, orient_images(qmap), "T1 (s)", cmap, vmax)

    plt.tight_layout()
    plt.show()

In [ ]:
show_image(t1_map, Colormap("lipari").to_mpl(), vmax)
show_image(t1_map_dicom, Colormap("lipari").to_mpl(), vmax)